# Imports

In [1]:
# Imports
import os
import sys
sys.path.append('../src/utils')
sys.path.append('../src')
import numpy as np
from tqdm import tqdm
import torch
import fvdb.nn as fvnn
import mesh_tools as mt
from models import unet as unetModels

from mesh_tools import count_parameters
from mesh_tools import plot_vdb

import igl
import trimesh
import fvdb

# Utils

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def fetch_numpy_values(grid: fvdb.GridBatch, arr: np.array, size:int):
    '''fetches values from a numpy array based on the ijk indices in the grid'''
    # ijk = grid.ijk.jdata.cpu().detach().numpy()+(size-1)//2
    ijk = grid.ijk.jdata.cpu().detach().numpy()
    
    if max(ijk[:, 0]) >= arr.shape[0] or max(ijk[:, 1]) >= arr.shape[1] or max(ijk[:, 2]) >= arr.shape[2]:
        # If indices are out of bounds, we can add the maximum value to the indices
        ijk = np.clip(ijk, 0, np.array(arr.shape) - 1)
        # print(f"Indices out of bounds. Clipping to max shape: {arr.shape}")
    
    values = arr[ijk[:, 0], ijk[:, 1], ijk[:, 2]]
    return torch.tensor(values, dtype=torch.float32, device=grid.device)

def scaled_sdf(sdf_arr: np.array, sdf_scaling_value: int):
    '''scales the SDF array by the threshold value'''
    return (sdf_scaling_value-1)*sdf_arr[:, None]

def sdf_to_vdb(sdf_arr: np.array, 
                mask: np.array, 
                size=33):

    '''Converts a SDF array to a VDBTensor with a given size and mask.'''
    print(f"Converting SDF to VDB with size {size} and mask {mask.shape}")
    sdf_scaling_value = (size-1)*2 + 1

    #  create a grid of the size without nomalize actual shape
    ijk_mesh_grid = mt.mesh_grid(size)
    ijk_mesh_grid = ijk_mesh_grid.reshape(size, size, size, 3)
    
    ijk = torch.tensor(ijk_mesh_grid[mask], 
                        dtype=torch.int, 
                        device=device)
    grid = fvdb.gridbatch_from_ijk(fvdb.JaggedTensor(ijk), 
                                    voxel_sizes=(1/(size-1)), 
                                    origins=torch.tensor([0, 0, 0], 
                                    device=device))
    
    sdf_values = fetch_numpy_values(grid, sdf_arr, size)
    sdf_values = scaled_sdf(sdf_values, sdf_scaling_value)
    return fvnn.VDBTensor(grid, grid.jagged_like(sdf_values))

def sdf_from_mesh(mesh_path: str, grid_n: int):
    '''Generates SDF from a mesh file using igl signed distance.'''
    v, f = igl.read_triangle_mesh(mesh_path)
    v = 2*mt.NDCnormalize(v)
    points = mt.mesh_grid(grid_n, True)
    sdf = igl.signed_distance(points, v, f)[0].reshape(grid_n, grid_n, grid_n)/2
    return sdf

# Params

`We will convert objs mesh to sdf at different resolution`

In [3]:
path = '../run/data/model_weights.pth'
gt_dir = '/data/workspaces/spanwar/dataset/thingi/GT_thingi'
file_name = os.listdir(gt_dir)[:1][0]
grid_size = 33 # [33, 65, 129] we can test on multiple grids
upsample_factor = 4

# Load Model

In [4]:
model = unetModels.FVDBUNetBase(
                in_channels=4,
                out_channels=1)

# load the trained model
model.load_state_dict(torch.load(path))
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

/tmp/ipykernel_3144433/2832388839.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path))


`Parameters`

In [5]:
count_parameters(model)

The model has 21.7M parameters


# Input Preparation

In [6]:
def get_all_shifted_positions(vdb_tensor, size, upsample_factor):
        mfg = torch.tensor(mt.mesh_grid(upsample_factor+1), device=vdb_tensor.device) - (upsample_factor//2)
        new_features = []
        for mg in mfg:
            org_ijk = vdb_tensor.grid.ijk.jdata
            ijk = (upsample_factor * org_ijk + mg).view(-1, 3)
            ijk = np.clip(ijk.cpu().detach().numpy(), 0, (size-1)*upsample_factor)
            ijk_vector = ijk - (org_ijk.cpu().detach().numpy() * upsample_factor)
            ijk_vector = ijk_vector / (upsample_factor // 2)  # Normalize to values between -1 and 1
            ijk_vector = torch.tensor(ijk_vector, dtype=torch.float32, device=vdb_tensor.device)
            new_features.append(torch.cat([vdb_tensor.data.jdata, ijk_vector], axis=-1))
        return new_features
    
def prepare_all_inputs(sdf_numpy, grid_size, upsample_factor=4):
    '''prepare voxel 4D input: SDF+displacement'''
    mask_threshold = grid_size*2+1
    mask = mt.make_mask_close(sdf_numpy, mask_threshold)
    input_vdb = sdf_to_vdb(sdf_numpy, mask, grid_size)
    new_features = get_all_shifted_positions(input_vdb, grid_size, upsample_factor)
    all_inputs = []
    for feature in new_features:
        all_inputs.append(fvnn.VDBTensor(input_vdb.grid,
                                        input_vdb.grid.jagged_like(feature)))
    return all_inputs

In [7]:
path = os.path.join(gt_dir, file_name)
sdf_numpy = sdf_from_mesh(path, grid_size)
all_inputs = prepare_all_inputs(sdf_numpy, grid_size, upsample_factor)

  o ./models_NDC_norm/GT_thingi/75655.obj 


Converting SDF to VDB with size 33 and mask (33, 33, 33)


# Prediction

In [8]:
def get_predictions(all_inputs,
                    input_size,
                    upsample_factor, 
                    model,
                    is_large=False,
                    BATCH_SIZE=20):

    sdf_scaling = (input_size-1)*2+1
    upsampled_sdf_size = ((input_size - 1) * upsample_factor) + 1
    sdf = np.full((upsampled_sdf_size, 
                    upsampled_sdf_size, 
                    upsampled_sdf_size), 100.0)
        
    num_samples = len(all_inputs)

    pred_values_list = []
    pred_ijk_list = []
    vector_list = []

    # model on device
    model.to(device)
    model.eval()

    with torch.no_grad(): 
        if is_large:
            for start in range(0, num_samples, BATCH_SIZE):
                end = min(start + BATCH_SIZE, num_samples)
                batch_vdb = fvdb.jcat(all_inputs[start:end])
                vector_list.append(batch_vdb.jdata[:, 1:4].cpu().detach().numpy())
                batch_vdb = batch_vdb.cuda()
                batch_pred = model(batch_vdb)
                pred_values_list.append(batch_pred.jdata.detach().cpu().numpy().squeeze())
                pred_ijk_list.append(batch_pred.grid.ijk.jdata.cpu().detach().numpy())

            pred_values = np.concatenate(pred_values_list, axis=0)
            pred_ijk = np.concatenate(pred_ijk_list, axis=0)
            vector = np.concatenate(vector_list, axis=0)
            pred_ijk = (pred_ijk)*upsample_factor + (vector*(upsample_factor//2)).astype(int)
        else:
            batch_vdb = fvdb.jcat(all_inputs)
            batch_pred = model(batch_vdb)

            vector = batch_vdb.jdata[:, 1:4].cpu().detach().numpy()
            pred_values = batch_pred.jdata.detach().cpu().numpy().squeeze()
            pred_ijk = batch_pred.grid.ijk.jdata.cpu().detach().numpy()
            pred_ijk = (pred_ijk)*upsample_factor + (vector*(upsample_factor//2)).astype(int)

    # means predictions
    flat_idx = np.ravel_multi_index(pred_ijk.T, sdf.shape)  # (N,)

    sum_arr = np.zeros(sdf.size, dtype=np.float32)
    cnt_arr = np.zeros(sdf.size, dtype=np.int64)

    np.add.at(sum_arr, flat_idx, pred_values)     # accumulate sums per voxel
    np.add.at(cnt_arr, flat_idx, 1)               # accumulate counts per voxel

    mask = cnt_arr > 0
    mean_arr = np.zeros_like(sum_arr, dtype=np.float32)
    mean_arr[mask] = sum_arr[mask] / cnt_arr[mask]

    sdf.flat[mask] = mean_arr[mask] 

    sdf_mask = np.abs(sdf) < 100
       
    # create a fvdb tensor from the sdf
    up_ijk = fvdb.JaggedTensor(torch.tensor(np.array(np.where(sdf_mask)).T))
    up_grid = fvdb.gridbatch_from_ijk(
            up_ijk,
            voxel_sizes=(1/(upsampled_sdf_size-1)),
            origins=torch.tensor([0, 0, 0])
        )
    up_ijk = up_grid.ijk.jdata.cpu().detach().numpy()
    up_values = sdf[up_ijk[:, 0], up_ijk[:, 1], up_ijk[:, 2]]
    # descale the sdf values
    up_values = up_values * (1/(sdf_scaling-1))
    up_tensor = fvnn.VDBTensor(up_grid,
                                up_grid.jagged_like(torch.tensor(up_values)))
    return up_tensor

In [9]:
output_vdb = get_predictions(all_inputs,
                    input_size=grid_size,
                    upsample_factor=upsample_factor,
                    model=model,
                    is_large=False)

# Plot

In [11]:
plot_vdb(output_vdb)

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.4992210…

# Save Mesh

In [ ]:
# save mesh
v, f, _ = output_vdb.grid.marching_cubes(output_vdb.data)
v=v.jdata.cpu().detach().numpy()
f=f.jdata.cpu().detach().numpy()
mesh = trimesh.Trimesh(vertices=v, faces=f)
mesh.export('predicted_mesh.obj')